In [20]:
%matplotlib qt
import mne

#mne.viz.set_3d_backend('pyvistaqt')
from mne.coreg import Coregistration
from mne.io import read_info


import numpy as np
#%matplotlib qt
import matplotlib
#matplotlib.use('qt5agg')  # Or any other backend you want to use

import matplotlib.pyplot as plt

import pandas as pd 
import os
from os.path import join as pathjoin
from pathlib import Path

import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from time import time

from autoreject import AutoReject

import glob

import psutil
import gc
from time import time

mne.set_log_level('INFO')

import json

from mne.channels import read_dig_polhemus_isotrak  # Función para leer archivos .pos

import re
# from mne.minimum_norm import apply_inverse, make_inverse_operator
#este codigo lo dejo comentado para acostumbrarme a su suso

import brainiak
from brainiak.isc import isc 

import statsmodels
from statsmodels.stats.multitest import multipletests

import pickle

In [21]:
try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

In [22]:
#subjects = subj[:9]

subjects = []

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subjects.append(subdirectorio.name)

print(subjects)



['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

In [23]:
def print5(*args):
    print(*(f"{x:.5f}" if isinstance(x, float) else x for x in args))

channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

channels_mag=channels_mag.tolist()
print5(channels_mag)
indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels


['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

In [24]:
valid_arrays = []
valid_subjects = []

n_subjects=len(subjects)
# Condiciones
for cond in ["zinnen", "woorden"]:
    
    valid_arrays = []
    valid_subjects = []

    for i in range(n_subjects):
        subj = subjects[i]
        file_path = evoked_path / f"{subj}_evoked_{cond}_{layer_script}-ave.fif"

        try:
            evokeds = mne.read_evokeds(file_path)
            evoked = evokeds[0]
            data_subj = evoked.pick("mag", exclude="bads").data  # (n_dipoles, n_times)
            data_subj_swapped = data_subj.T  # (n_times, n_dipoles)

            # Validar que los datos no sean basura
            if np.all(data_subj_swapped == 0) or np.isnan(data_subj_swapped).all():
                print(f"{subj} tiene solo ceros o NaNs, se omite.")
                continue

            valid_arrays.append(data_subj_swapped)
            valid_subjects.append(subj)

        except FileNotFoundError:
            print(f"Archivo no encontrado para {subj}: {file_path}")
            continue
        except Exception as e:
            print(f"Error al procesar {subj}: {e}")
            continue


    array = np.stack(valid_arrays, axis=2)  # (n_times, n_dipoles, n_subjects)
    print(f"Array ISC creado con forma {array.shape} ({len(valid_subjects)} sujetos válidos)")

    iscs = isc(data=array, pairwise=False, summary_statistic=None, tolerate_nans=True)
    iscs_statistics=brainiak.isc.compute_summary_statistic(iscs, summary_statistic='mean', axis=0)

    ##bootstrap zinnen gives you the isc values, the confidence intervals, the p values and the distribution of the isc values
    ## so  isc=iscs_bootstrap [0]

    iscs_bootstrap, ci,p, distribution= brainiak.isc.bootstrap_isc(iscs, 
    pairwise=False, summary_statistic='median', n_bootstraps=1000, 
    ci_percentile=95, side='right', random_state=None)


    print(f"iscs_statistics, {iscs_statistics}")
    print(f"iscs_bootstrap: {iscs}")
    print(f"ci: {ci}")
    print(f"p: {p}")
    print(f"distribution: {distribution}")
    print(f"iscs_statistics: {iscs_statistics}")

    threshold = 0.05

    # Encontrar los índices (canales) donde p[0] es menor que 0.05
    significant_channels = np.where(p < threshold)[0]

    # Contar cuántos canales cumplen la condición
    num_significant_channels = len(significant_channels)

    # Imprimir los canales significativos
    print(f"Canales significativos {cond} (p < 0.05):", significant_channels)
    print(f"Número total de canales significativos general en {cond}:", num_significant_channels)


    # Aplicar la corrección de Benjamini-Hochberg (FDR)
    _, p_adjusted, _, _ = multipletests(p, method='fdr_bh')
    significant_channels_adjusted = np.where(p_adjusted < threshold)[0]

    # Mostrar cuántos canales siguen siendo significativos
    num_significant_channels_adjusted = len(significant_channels_adjusted)
    print(f"Canales significativos {cond} después de FDR-BH: {significant_channels_adjusted} de {len(channels_mag)}")
    print(f"Número de Canales significativos {cond} después de FDR-BH: {num_significant_channels_adjusted} de {len(channels_mag)}")
    dict = {
        f"iscs_cond_{cond}": iscs,
        f"iscs_statistics_{cond}": iscs_statistics,
        f"iscs_bootstrap_{cond}": iscs_bootstrap,
        f"ci_{cond}": ci,
        f"p_{cond}": p,
        f"distribution_{cond}": distribution,
        f"significant_channels_{cond}": significant_channels,
        f"num_significant_channels_{cond}": num_significant_channels,
        f"p_adjusted_{cond}": p_adjusted,
        f"significant_channels_adjusted_{cond}": significant_channels_adjusted,
        f"num_significant_channels_adjusted_{cond}": num_significant_channels_adjusted
    }
    
    if cond=="zinnen":
        dict_zinnen = dict
    elif cond=="woorden":
        dict_woorden = dict
    
    # Definir la ruta de guardado
    pickle_path = ISC_path / f"dict_{cond}_{layer_script}.pkl"

    # Guardar el diccionario
    with open(pickle_path, 'wb') as f:
        pickle.dump(dict, f)

    print(f"Diccionario guardado en: {pickle_path}")
    
    dict={}
    

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1001_evoked_zinnen_block-ave.fif ...


    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_zinnen)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1002_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_zinnen)
        0 CTF compensation matrices available
        nave = 22 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Archivo no encontrado para sub-V1003: g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1003_evoked_zinnen_block-ave.fif
Archivo no encontrado para sub-V1004: g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1004_evoked_zinnen_block-ave.fif
Reading g:\MOUS_204\MOUS_visual\

In [25]:
dict_zinnen

{'iscs_cond_zinnen': array([[ 0.00217445,  0.01426722,  0.02222028, ...,  0.00034647,
          0.01334904,  0.01268053],
        [-0.02116611, -0.00856475, -0.01106148, ...,  0.02163695,
         -0.00170158,  0.03141469],
        [ 0.04095289,  0.04218928,  0.03148259, ...,  0.01365321,
          0.01213569,  0.01214062],
        ...,
        [ 0.03009538,  0.03737349,  0.01985852, ..., -0.00470247,
         -0.0318483 ,  0.02186062],
        [ 0.04577013,  0.07167906,  0.09133272, ...,  0.01803015,
          0.02038361,  0.04561802],
        [ 0.0450941 ,  0.05611229,  0.04377188, ...,  0.00966532,
          0.0322095 , -0.00957418]], shape=(18, 273)),
 'iscs_statistics_zinnen': array([ 1.20033830e-02,  1.63677789e-02,  1.09560693e-02,  1.24044109e-02,
         1.59648176e-02,  2.46500187e-02,  2.43281809e-02,  4.80481016e-03,
         3.84440139e-03,  1.59986108e-02,  3.01899321e-02,  3.33009656e-02,
         2.22590870e-02,  3.08512250e-02,  1.01627638e-02,  2.67626076e-02,
      

In [26]:
dict_woorden

{'iscs_cond_woorden': array([[ 0.0060574 , -0.00968741, -0.01832319, ..., -0.02193594,
          0.00110158,  0.04220487],
        [-0.01445525,  0.00564721,  0.02178404, ...,  0.02300709,
          0.0372561 , -0.05279757],
        [ 0.00275889, -0.00188413, -0.00340268, ...,  0.02897904,
          0.0106923 , -0.00751706],
        ...,
        [-0.04086999, -0.05851545, -0.05318098, ...,  0.01721802,
         -0.0296234 ,  0.02046599],
        [ 0.00087966,  0.0083515 , -0.01166431, ..., -0.01532284,
         -0.00355893, -0.02049717],
        [ 0.0020248 , -0.00755133, -0.01246967, ..., -0.04273916,
         -0.03328276, -0.01237417]], shape=(18, 273)),
 'iscs_statistics_woorden': array([ 4.90188542e-03,  3.85037309e-03, -1.17520170e-03, -7.18567852e-03,
        -5.66032481e-03,  8.94617310e-03,  1.15790774e-02, -9.56254923e-05,
        -3.26829939e-03,  5.59454668e-04,  1.38542715e-02,  1.74904604e-02,
         1.11279311e-02,  1.90096233e-02,  5.95703282e-03,  1.99808382e-02,
    